In [0]:
# Incremental Ingestion Demo (Bronze Layer) using Auto Loader

#   - Day1 ingestion
#   - Day2 ingestion (new + updated + duplicate records)
#   - Day3 ingestion (schema evolution)

In [0]:
# Base incremental folder
incremental_path = "/Volumes/ecommerce/bronze/raw/incremental/"

# Checkpoint folder for Auto Loader
checkpoint_path = "/Volumes/ecommerce/bronze/raw/checkpoints/incremental_orders"

print("Incremental Path:", incremental_path)
print("Checkpoint Path:", checkpoint_path)

#### Preview incremental files (batch read)

In [0]:
preview_df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv("/Volumes/ecommerce/bronze/raw/incremental/*/*.csv"))

preview_df = preview_df.withColumn("source_file", preview_df["_metadata.file_path"])
display(preview_df)

##### Start Auto Loader Stream (Bronze Incremental Table)

In [0]:
from pyspark.sql import functions as F

incremental_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/ecommerce/bronze/raw/schema/incremental_orders"
    )
    .load("/Volumes/ecommerce/bronze/raw/incremental/*/")
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("bronze_ingested_at", F.current_timestamp())
)

In [0]:
incremental_stream_df.writeStream \
    .format("delta") \
    .option("cloudFiles.schemaLocation", "/Volumes/ecommerce/bronze/raw/schema/incremental_orders") \
    .option("checkpointLocation", "/Volumes/ecommerce/bronze/raw/checkpoints/incremental_orders") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("mergeSchema", "true") \
    .table("ecommerce.bronze.incremental_orders")


##### Bronze Ingestion Validation



In [0]:
# Load Bronze incremental table
bronze_df = spark.table("ecommerce.bronze.incremental_orders")

print("Total Records in Bronze Incremental Table:", bronze_df.count())
display(bronze_df.orderBy("order_date"))

##### Delta History (Bronze Table)

In [0]:
spark.sql("""
DESCRIBE HISTORY ecommerce.bronze.incremental_orders
""").show(truncate=False)


In [0]:
# %sql
# DROP TABLE IF EXISTS ecommerce.bronze.incremental_orders;

# DROP TABLE IF EXISTS ecommerce.silver.incremental_orders_silver;

# DROP TABLE IF EXISTS ecommerce.gold.daily_revenue;
# DROP TABLE IF EXISTS ecommerce.gold.category_revenue;
# DROP TABLE IF EXISTS ecommerce.gold.top_customers;